In [0]:
# Silver Customer Sales Pipeline 

# Join bronze_customers and bronze_sales on customer_id 

# Include product info, customer location, loyalty segment 

# Create derived column: sales_band based on total_price 

 

In [0]:
from pyspark.sql.functions import col, when 

 # Configuration 

target_catalog = "dbacademy" 

target_schema = "healthcare" 

customers_table = f"{target_catalog}.{target_schema}.bronze_customers" 

sales_table = f"{target_catalog}.{target_schema}.bronze_sales" 

target_table = f"{target_catalog}.{target_schema}.silver_customer_sales" 

 

In [0]:
# Read bronze tables 

df_customers = spark.read.table(customers_table) 

df_sales = spark.read.table(sales_table) 

 

# Join both tables on customer_id 

df_joined = df_sales.alias("s").join( 

    df_customers.alias("c"), 

    col("s.customer_id") == col("c.customer_id"), 

    "inner" 

).select( 

    col("s.customer_id"), 

    col("s.customer_name"), 

    # Product information 

    col("s.product_name"), 

    col("s.product_category"), 

    col("s.product"), 

    col("s.order_date"), 

    col("s.total_price"), 

    # Customer location 

    col("c.state"), 

    col("c.city"), 

    col("c.region"), 

    col("c.district"), 

    # Loyalty segment 

    col("c.loyalty_segment") 

) 

 

# Create derived column: sales_band 

# total_price < 100 -> LOW 

# total_price 100-500 -> MEDIUM 

# total_price > 500 -> HIGH 

df_with_band = df_joined.withColumn( 

    "sales_band", 

    when(col("total_price") < 100, "LOW") 

    .when((col("total_price") >= 100) & (col("total_price") <= 500), "MEDIUM") 

    .otherwise("HIGH") 

) 


In [0]:

# Save as silver_customer_sales 

df_with_band.write.mode("overwrite").saveAsTable(target_table) 



In [0]:

print(f"Successfully saved silver_customer_sales to {target_table}") 
print(f"Record count: {spark.read.table(target_table).count()}") 